<a href="https://colab.research.google.com/github/BandanaSingha24/Integrated_Multiomic_Survival_Model/blob/main/01_Python_Foundation%20_for_Bioinformatics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np

 #step 1 : lodind the Clinical data

# 1.1 Load the patient clinical data
# comment='#' skips the metadata/header rows starting with a hash symbol
patient_df = pd.read_csv('data_clinical_patient.txt', sep='\t', comment='#')

# 1.2 Load the sample clinical data
sample_df = pd.read_csv('data_clinical_sample.txt', sep='\t', comment='#')

#  Display the first 5 rows of both DataFrames to verify the data
print("--- Patient Data Overview ---")
display(patient_df.head())

print("\n--- Sample Data Overview ---")
display(sample_df.head())

--- Patient Data Overview ---


,PATIENT_ID,LYMPH_NODES_EXAMINED_POSITIVE,NPI,CELLULARITY,CHEMOTHERAPY,COHORT,ER_IHC,HER2_SNP6,HORMONE_THERAPY,INFERRED_MENOPAUSAL_STATE,...,OS_STATUS,CLAUDIN_SUBTYPE,THREEGENE,VITAL_STATUS,LATERALITY,RADIO_THERAPY,HISTOLOGICAL_SUBTYPE,BREAST_SURGERY,RFS_MONTHS,RFS_STATUS
0,MB-0000,10.0,6.044,NaN,NO,1.0,Positve,NEUTRAL,YES,Post,...,0:LIVING,claudin-low,ER-/HER2-,Living,Right,YES,Ductal/NST,MASTECTOMY,140.500000,0:Not Recurred
1,MB-0002,0.0,4.020,High,NO,1.0,Positve,NEUTRAL,YES,Pre,...,0:LIVING,LumA,ER+/HER2- High Prolif,Living,Right,YES,Ductal/NST,BREAST CONSERVING,84.633333,0:Not Recurred
2,MB-0005,1.0,4.030,High,YES,1.0,Positve,NEUTRAL,YES,Pre,...,1:DECEASED,LumB,NaN,Died of Disease,Right,NO,Ductal/NST,MASTECTOMY,153.300000,1:Recurred
3,MB-0006,3.0,4.050,Moderate,YES,1.0,Positve,NEUTRAL,YES,Pre,...,0:LIVING,LumB,NaN,Living,Right,YES,Mixed,MASTECTOMY,164.933333,0:Not Recurred
4,MB-0008,8.0,6.080,High,YES,1.0,Positve,NEUTRAL,YES,Post,...,1:DECEASED,LumB,ER+/HER2- High Prolif,Died of Disease,Right,YES,Mixed,MASTECTOMY,18.800000,1:Recurred



--- Sample Data Overview ---


,PATIENT_ID,SAMPLE_ID,CANCER_TYPE,CANCER_TYPE_DETAILED,ER_STATUS,HER2_STATUS,GRADE,ONCOTREE_CODE,PR_STATUS,SAMPLE_TYPE,TUMOR_SIZE,TUMOR_STAGE,TMB_NONSYNONYMOUS
0,MB-0000,MB-0000,Breast Cancer,Breast Invasive Ductal Carcinoma,Positive,Negative,3.0,IDC,Negative,Primary,22.0,2.0,0.000000
1,MB-0002,MB-0002,Breast Cancer,Breast Invasive Ductal Carcinoma,Positive,Negative,3.0,IDC,Positive,Primary,10.0,1.0,2.615035
2,MB-0005,MB-0005,Breast Cancer,Breast Invasive Ductal Carcinoma,Positive,Negative,2.0,IDC,Positive,Primary,15.0,2.0,2.615035
3,MB-0006,MB-0006,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,Positive,Negative,2.0,MDLC,Positive,Primary,25.0,2.0,1.307518
4,MB-0008,MB-0008,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,Positive,Negative,3.0,MDLC,Positive,Primary,40.0,2.0,2.615035


In [5]:
#step 2:Merging the clinical data

# 2.1 Merge with explicit suffixes to avoid confusion later
combined_df = pd.merge(patient_df, sample_df, on='PATIENT_ID', how='inner', suffixes=('_patient', '_sample'))

# Professor-Level Sanity Check: Ensure unique samples per patient if required
unique_patients = combined_df['PATIENT_ID'].nunique()
total_rows = combined_df.shape[0]
if unique_patients != total_rows:
    print(f"Warning: Found {total_rows - unique_patients} multi-sample patient entries. Check for duplicates!")

# Display the first 5 rows of the combined dataframe to verify the merge
print("\n--- Combined Data Overview ---")
display(combined_df.head())


--- Combined Data Overview ---


,PATIENT_ID,LYMPH_NODES_EXAMINED_POSITIVE,NPI,CELLULARITY,CHEMOTHERAPY,COHORT,ER_IHC,HER2_SNP6,HORMONE_THERAPY,INFERRED_MENOPAUSAL_STATE,...,CANCER_TYPE_DETAILED,ER_STATUS,HER2_STATUS,GRADE,ONCOTREE_CODE,PR_STATUS,SAMPLE_TYPE,TUMOR_SIZE,TUMOR_STAGE,TMB_NONSYNONYMOUS
0,MB-0000,10.0,6.044,NaN,NO,1.0,Positve,NEUTRAL,YES,Post,...,Breast Invasive Ductal Carcinoma,Positive,Negative,3.0,IDC,Negative,Primary,22.0,2.0,0.000000
1,MB-0002,0.0,4.020,High,NO,1.0,Positve,NEUTRAL,YES,Pre,...,Breast Invasive Ductal Carcinoma,Positive,Negative,3.0,IDC,Positive,Primary,10.0,1.0,2.615035
2,MB-0005,1.0,4.030,High,YES,1.0,Positve,NEUTRAL,YES,Pre,...,Breast Invasive Ductal Carcinoma,Positive,Negative,2.0,IDC,Positive,Primary,15.0,2.0,2.615035
3,MB-0006,3.0,4.050,Moderate,YES,1.0,Positve,NEUTRAL,YES,Pre,...,Breast Mixed Ductal and Lobular Carcinoma,Positive,Negative,2.0,MDLC,Positive,Primary,25.0,2.0,1.307518
4,MB-0008,8.0,6.080,High,YES,1.0,Positve,NEUTRAL,YES,Post,...,Breast Mixed Ductal and Lobular Carcinoma,Positive,Negative,3.0,MDLC,Positive,Primary,40.0,2.0,2.615035


In [6]:
# step 3: Data Cleaning

# 1. Standardize string categories (e.g., turn 'Pos' into 'Positive')
combined_df['PR_STATUS'] = combined_df['PR_STATUS'].replace({'Pos': 'Positive', 'Neg': 'Negative'})
combined_df['PR_STATUS'] = combined_df['PR_STATUS'].str.strip().str.capitalize()

# 2. Check for missing clinical data before proceeding
print(combined_df[['PR_STATUS', 'TUMOR_SIZE', 'TUMOR_STAGE']].isnull().sum())

PR_STATUS      529
TUMOR_SIZE     149
TUMOR_STAGE    721
dtype: int64


In [7]:
# step 4: Imputation of missing Clinical data

# 1. Impute numerical tumor size with median
combined_df['TUMOR_SIZE'] = combined_df['TUMOR_SIZE'].fillna(combined_df['TUMOR_SIZE'].median())

# 2. Impute categorical columns with an 'Unknown' flag
combined_df['PR_STATUS'] = combined_df['PR_STATUS'].fillna('Unknown')
combined_df['TUMOR_STAGE'] = combined_df['TUMOR_STAGE'].fillna('Unknown')

# 3. Final Sanity Check (Should print all 0s)
print("Remaining missing values:")
print(combined_df[['PR_STATUS', 'TUMOR_SIZE', 'TUMOR_STAGE']].isnull().sum())

Remaining missing values:
PR_STATUS      0
TUMOR_SIZE     0
TUMOR_STAGE    0
dtype: int64


In [11]:
combined_df.to_csv('md-1_processed_clinical_data.csv', index=False)

from google.colab import files
files.download('md-1_processed_clinical_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>